In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Datasets/7817_1.csv')

In [7]:
df = df.dropna(subset=['reviews.text', 'reviews.rating'])

In [8]:
df['sentiment'] = df['reviews.rating'].apply(lambda x: 1 if x > 3 else 0)

/tmp/ipykernel_28294/3470601538.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['reviews.rating'].apply(lambda x: 1 if x > 3 else 0)


In [9]:
texts = df['reviews.text'].values
labels = df['sentiment'].values

In [10]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

In [11]:
max_features = 10000
max_len = 150

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(X_train_text)

X_train = tokenizer.texts_to_sequences(X_train_text)
X_test = tokenizer.texts_to_sequences(X_test_text)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [12]:
model = Sequential()
model.add(Embedding(input_dim=max_features, output_dim=64, input_length=max_len))
model.add(GRU(64, return_sequences=False))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [13]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [14]:
print("Training model...")
history = model.fit(X_train, y_train,epochs=5,batch_size=64,validation_split=0.1)

Training model...
Epoch 1/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 109ms/step - accuracy: 0.7967 - loss: 0.6480 - val_accuracy: 0.8105 - val_loss: 0.5730
Epoch 2/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step - accuracy: 0.8274 - loss: 0.4973 - val_accuracy: 0.8105 - val_loss: 0.4360
Epoch 3/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.8286 - loss: 0.3846 - val_accuracy: 0.8632 - val_loss: 0.3868
Epoch 4/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 138ms/step - accuracy: 0.8593 - loss: 0.3270 - val_accuracy: 0.8632 - val_loss: 0.3486
Epoch 5/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.8830 - loss: 0.2351 - val_accuracy: 0.8947 - val_loss: 0.3160


In [15]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8814 - loss: 0.3820
Test Accuracy: 88.14%
